# Return and Risk

Return and risk are the basic language of portfolio management. This notebook moves from single-asset price data to return, volatility, covariance, and drawdown.

Abbreviations used in this notebook:

- **CAGR**: Compound Annual Growth Rate.
- **SD**: Standard Deviation, a common volatility measure.
- **VaR**: Value at Risk, a downside loss threshold.
- **CHF**: Swiss franc, used only as an illustrative currency.

## 1. Intuition

Investors care about both reward and uncertainty. Return measures how wealth grows. Risk measures how uncertain, unstable, or painful that growth can be.

The key shift is from price levels to returns. Portfolio math is usually done on returns because returns are comparable across assets with different price levels.

## 2. Mathematics

**Simple return:**

$$
R_t = \frac{P_t}{P_{t-1}} - 1
$$

Where:

- $R_t$ = simple return at time $t$
- $P_t$ = price at time $t$
- $P_{t-1}$ = price in the previous period
- $t$ = time period index

**Cumulative return:**

$$
R_{cum} = \prod_t (1 + R_t) - 1
$$

Where:

- $R_t$ = simple return at time $t$
- $R_{cum}$ = cumulative return across the period
- $t$ = time period index

**Annualized volatility:**

$$
\sigma_{annual} = \sigma_{daily} \sqrt{252}
$$

Where:

- $\sigma_{annual}$ = annualized volatility
- $\sigma_{daily}$ = daily volatility
- $252$ = approximate number of trading days in one year

**Covariance:**

$$
Cov(R_i, R_j) = E[(R_i - \mu_i)(R_j - \mu_j)]
$$

Where:

- $Cov(R_i, R_j)$ = covariance between returns of assets $i$ and $j$
- $R_i$ = return of asset $i$
- $R_j$ = return of asset $j$
- $\mu_i$ = expected return of asset $i$
- $\mu$ = vector of expected returns or expected drift

**Historical VaR at 5 percent:**

$$
VaR_{5\%} = -Percentile(R, 5\%)
$$

Where:

- $VaR$ = value at risk, a downside loss threshold
- $Percentile(R, 5\%)$ = fifth percentile of the return distribution

## 3. Implementation

We generate a synthetic multi-asset universe and calculate return and risk statistics.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "03_portfolio_management" / "portfolio_utils.py"
spec = importlib.util.spec_from_file_location("portfolio_utils", helper_path)
portfolio_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(portfolio_utils)

plt.style.use("seaborn-v0_8-whitegrid")

returns = portfolio_utils.generate_synthetic_returns(periods=504, seed=321)
prices = portfolio_utils.returns_to_prices(returns)
assets = returns.columns.tolist()

returns.head()

In [ ]:
risk_table = pd.DataFrame({
    "annualized_return": portfolio_utils.annualized_return(returns),
    "annualized_volatility": portfolio_utils.annualized_volatility(returns),
    "daily_var_5pct": -returns.quantile(0.05),
    "best_day": returns.max(),
    "worst_day": returns.min(),
})

risk_table.round(4)

## 4. Visualization

The first portfolio diagnostics are usually indexed prices, return distributions, and a correlation matrix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

prices.plot(ax=axes[0])
axes[0].set_title("Indexed Asset Prices")
axes[0].set_ylabel("Index value")
axes[0].legend(loc="upper left")

returns["Global Equity"].hist(bins=35, ax=axes[1], color="#2f6f8f", edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Daily Return Distribution: Global Equity")
axes[1].set_xlabel("Daily return")
axes[1].xaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")

plt.tight_layout()
plt.show()

In [ ]:
correlation = returns.corr()

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(correlation.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(assets)))
ax.set_xticklabels(assets, rotation=35, ha="right")
ax.set_yticks(range(len(assets)))
ax.set_yticklabels(assets)
ax.set_title("Asset Return Correlation")

for row in range(len(assets)):
    for col in range(len(assets)):
        ax.text(col, row, f"{correlation.values[row, col]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

## 5. Application

Risk is not one number. Volatility captures variation, VaR captures tail loss thresholds, and correlation captures diversification potential.

In [ ]:
equal_weights = np.repeat(1 / len(assets), len(assets))
portfolio_returns = portfolio_utils.portfolio_series(returns, equal_weights)
dd = portfolio_utils.drawdown(portfolio_returns)

portfolio_summary = pd.Series({
    "annualized_return": portfolio_utils.annualized_return(portfolio_returns),
    "annualized_volatility": portfolio_utils.annualized_volatility(portfolio_returns),
    "daily_var_5pct": -portfolio_returns.quantile(0.05),
    "max_drawdown": dd["drawdown"].min(),
})

portfolio_summary.to_frame("equal_weight_portfolio")

## 6. Reflection

- Returns make assets comparable.
- Volatility is useful, but it is not the only risk measure.
- Correlation is the foundation of diversification.
- Drawdown captures investor pain better than volatility alone.

Questions to answer after running the notebook:

1. Which asset had the highest return?
2. Which asset had the highest volatility?
3. Which pair of assets offers the most diversification?
4. Why can a lower-volatility portfolio still have painful drawdowns?